In [ ]:
customers = spark.table("olist_.silver_customers")

dim_customers = customers.select(
    "customer_id",
    "customer_unique_id",
    "customer_city",
    "customer_state"
)

dim_customers.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("olist_.gold_dim_customers")

In [ ]:
products = spark.table("olist_.silver_products")
category = spark.table("olist_.silver_category_translation")

dim_products = (
    products.alias("p")
    .join(category.alias("c"),
          "product_category_name",
          "left")
    .select(
        "p.product_id",
        "p.product_category_name",
        "c.product_category_name_english",
        "p.product_weight_g",
        "p.product_length_cm",
        "p.product_height_cm",
        "p.product_width_cm"
    )
)

dim_products.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("olist_.gold_dim_products")

In [ ]:
sellers = spark.table("olist_.silver_sellers")

dim_sellers = sellers.select(
    "seller_id",
    "seller_city",
    "seller_state"
)

dim_sellers.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("olist_.gold_dim_sellers")

In [ ]:
orders = spark.table("olist_.silver_orders")
items = spark.table("olist_.silver_order_items")
payments = spark.table("olist_.silver_payments")

fact_orders = (
    orders.alias("o")
    .join(items.alias("i"), "order_id")
    .join(payments.alias("p"), "order_id")
    .select(
        "o.order_id",
        "o.customer_id",
        "o.order_status",
        "o.order_purchase_timestamp",
        "i.product_id",
        "i.seller_id",
        "i.price",
        "i.freight_value",
        "p.payment_type",
        "p.payment_installments",
        "p.payment_value"
    )
)

fact_orders.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("olist_.gold_fact_orders")